# Results Analysis — Strategy Comparison & Final Figures

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import yaml
from pathlib import Path

from src.environment.trading_env import TradingEnv
from src.agents.dqn_agent import load_agent, run_episode
from src.agents import buy_and_hold, ma_crossover
from src.evaluation.metrics import compute_all
from src.evaluation.backtester import run_all, compare_strategies, backtest_baseline

with open("../configs/dqn_config.yaml") as f:
    cfg = yaml.safe_load(f)
ENV_CFG = cfg["environment"]

os.makedirs("../results/figures", exist_ok=True)
os.makedirs("../results/tables",  exist_ok=True)

# Normalised data for DQN (TradingEnv), raw for baselines
df_test_norm = pd.read_csv("../data/normalized/test.csv", index_col=0, parse_dates=True)
df_val_norm  = pd.read_csv("../data/normalized/val.csv",  index_col=0, parse_dates=True)
df_test      = pd.read_csv("../data/features/test.csv",   index_col=0, parse_dates=True)
df_val       = pd.read_csv("../data/features/val.csv",    index_col=0, parse_dates=True)

print(f"Val : {len(df_val)} rows  |  Test: {len(df_test)} rows")

## 1. Load Best DQN Model & Run All Strategies on Test Set

In [ ]:
best_model_path = "../models/best/best_model.zip"
assert Path(best_model_path).exists(), (
    "Best model not found. Run notebook 04 first to train the DQN."
)

env_test = TradingEnv(df_test_norm, ENV_CFG)
dqn_agent = load_agent(best_model_path, env_test)

# Run all strategies on the test set
results, table = run_all(
    df_raw=df_test,
    df_norm=df_test_norm,
    agent=dqn_agent,
    env_config=ENV_CFG,
    save_dir="../results/traces/test",
)

print("=" * 50)
print("FINAL TEST SET RESULTS")
print("=" * 50)
print(table.to_string())

# Save metrics table as CSV
table.to_csv("../results/tables/test_metrics.csv")
print("\nSaved → ../results/tables/test_metrics.csv")

## 2. Portfolio Value Chart — All Strategies vs Test Set Price

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True,
                         gridspec_kw={"height_ratios": [2, 1]})
ax1, ax2 = axes
dates = df_test.index

# Normalise portfolio values to the initial balance for fair comparison
# (DQN portfolio is in normalised price units, baselines in USD)
ib = ENV_CFG["initial_balance"]

colors = {"Buy-and-Hold": "steelblue", "MA-Crossover": "darkorange", "DQN": "crimson"}
for name, trace in results.items():
    pv = trace["portfolio_value"]
    # Scale so all start at initial_balance for visual comparison
    scaled = pv / pv.iloc[0] * ib
    ax1.plot(dates, scaled, label=name, linewidth=2, color=colors[name])

ax1.axhline(ib, color="grey", linewidth=1, linestyle=":", label="Initial balance")
ax1.set_ylabel("Portfolio Value (relative, USDT)")
ax1.set_title("Strategy Comparison — Test Set (Feb 2025)")
ax1.legend()
ax1.grid(alpha=0.3)

# Bottom: BTC/USDT close price
ax2.plot(dates, df_test["close"], color="black", linewidth=1.5)
ax2.set_ylabel("BTC Close (USDT)")
ax2.set_xlabel("Date")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("../results/figures/05_test_portfolio_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Metrics Bar Chart — Side-by-Side Comparison

In [ ]:
metrics_to_plot = ["Total Return", "Sharpe Ratio", "Sortino Ratio",
                   "Calmar Ratio", "Max Drawdown", "Win Rate"]
plot_table = table.loc[metrics_to_plot]

n_metrics   = len(metrics_to_plot)
n_strategies = len(plot_table.columns)
x = np.arange(n_metrics)
width = 0.25

fig, ax = plt.subplots(figsize=(14, 5))
for i, (strategy, color) in enumerate(zip(plot_table.columns, colors.values())):
    vals = plot_table[strategy].values.astype(float)
    bars = ax.bar(x + i * width, vals, width, label=strategy, color=color, alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f"{val:.2f}", ha="center", va="bottom", fontsize=8)

ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x + width)
ax.set_xticklabels(metrics_to_plot, rotation=15, ha="right")
ax.set_title("Strategy Metrics Comparison — Test Set (Feb 2025)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("../results/figures/05_metrics_bar_chart.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Drawdown Chart — Test Set

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
for name, trace in results.items():
    pv = trace["portfolio_value"]
    rolling_max = pv.cummax()
    drawdown = (pv / rolling_max - 1.0) * 100
    ax.plot(dates, drawdown, label=name, linewidth=1.5, color=colors[name])

ax.fill_between(dates, 0, 0, alpha=0)   # baseline
ax.axhline(0, color="grey", linewidth=0.8, linestyle=":")
ax.set_ylabel("Drawdown (%)")
ax.set_xlabel("Date")
ax.set_title("Drawdown — Test Set (Feb 2025)")
ax.legend()
ax.grid(alpha=0.3)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.1f}%"))
plt.tight_layout()
plt.savefig("../results/figures/05_drawdown.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. DQN Trade Actions on Test Set

In [ ]:
dqn_trace = results["DQN"]

# Align trace to df_test by POSITION (row order), not by index label.
# This is robust regardless of whether dqn_trace.index is integers or timestamps.
buy_mask  = dqn_trace["action"].values == 1   # numpy bool array, length = len(df_test)
sell_mask = dqn_trace["action"].values == 2

buy_dates   = dates[buy_mask]                  # DatetimeIndex from df_test
sell_dates  = dates[sell_mask]
buy_prices  = df_test["close"].values[buy_mask]
sell_prices = df_test["close"].values[sell_mask]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(dates, df_test["close"], color="black", linewidth=1.5, label="BTC/USDT Close")

ax.scatter(buy_dates,  buy_prices,  marker="^", color="green", s=120, zorder=5)
ax.scatter(sell_dates, sell_prices, marker="v", color="red",   s=120, zorder=5)

# Custom legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color="black", linewidth=1.5, label="BTC/USDT"),
    plt.scatter([], [], marker="^", color="green", s=80, label=f"Buy  ({buy_mask.sum()})"),
    plt.scatter([], [], marker="v", color="red",   s=80, label=f"Sell ({sell_mask.sum()})"),
]
ax.legend(handles=legend_elements, loc="upper left")
ax.set_ylabel("BTC/USDT (USDT)")
ax.set_xlabel("Date")
ax.set_title("DQN Trade Actions — Test Set (Feb 2025)")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("../results/figures/05_dqn_actions_test.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"DQN actions — Hold: {(dqn_trace['action']==0).sum()}  "
      f"Buy: {buy_mask.sum()}  "
      f"Sell: {sell_mask.sum()}")